# NanoGPT speedrun — T4 port (`nanogpt-colab-speedrun`)

Single-GPU port of [KellerJordan/modded-nanogpt](https://github.com/KellerJordan/modded-nanogpt) record #360 to a **Colab T4 (16 GB)**.
Target: **≤3.5 canonical-masked FineWeb val cross-entropy** (upstream: 3.28 on 8×H100).

**Before running:** Runtime → Change runtime type → T4 GPU. Then run the cells in order:

1. assert the environment (CUDA torch with Turing kernels)
2. install uv, clone the repo
3. build a `--system-site-packages` venv (reuses Colab's CUDA torch + bundled triton; never installs a second torch)
4. download the data (4×100M train tokens + val, ~800 MB)
5. smoke run (~2 min, small schedule)
6. full run (hours — keep the tab open)

Progress prints every 25 steps, validation every 150, full logs in `logs/<run_id>.txt`.

In [1]:
import torch

assert torch.cuda.is_available(), "No GPU runtime — set Runtime > Change runtime type > T4 GPU"
cap = torch.cuda.get_device_capability()
arch = torch.cuda.get_arch_list()
print(f"GPU: {torch.cuda.get_device_name(0)}  sm{cap[0]}{cap[1]}")
print(f"torch {torch.__version__}  CUDA {torch.version.cuda}  arch: {arch}")
# The port's Triton kernels are tuned for sm75 (T4). The torch build must contain it,
# and the current device must be served by this build (Q9 of the port plan).
assert "sm_75" in arch, f"torch build has no sm_75 (Turing) kernels: {arch}"
dev_arch = f"sm_{cap[0]}{cap[1]}"
assert dev_arch in arch, f"device {dev_arch} not in this build's arch list {arch}"
print("environment OK")

GPU: Tesla T4  sm75
torch 2.11.0+cu130  CUDA 13.0  arch: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
environment OK


In [2]:
import os
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
os.chdir("/content")  # absolute paths: an earlier %cd must not break re-runs
repo = "/content/nanogpt-colab-speedrun"
if not os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KingNish24/nanogpt-colab-speedrun.git", repo], check=True)
else:
    # Re-runs must never train stale code: hard-reset the existing clone to the remote tip.
    subprocess.run(["git", "-C", repo, "fetch", "--depth", "1", "origin"], check=True)
    subprocess.run(["git", "-C", repo, "reset", "--hard", "FETCH_HEAD"], check=True)
print("live commit:", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())
%cd /content/nanogpt-colab-speedrun

live commit: 2a3464f Stage 1 of the speed plan: wider CE chunks, compiled loss, 5-step logs
/content/nanogpt-colab-speedrun


In [3]:
# A venv layered on the Colab system environment: torch/triton come from the image,
# the repo's extras (tiktoken, datasets, ...) are installed into the venv.
!uv venv --system-site-packages .venv --clear
!uv pip install --python .venv/bin/python -r requirements.txt
!.venv/bin/python -c "import torch, triton; print('venv OK: torch', torch.__version__, '| triton', triton.__version__, '|', torch.cuda.get_device_name(0))"

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


Resolved 38 packages in 201ms                                        
Installed 38 packages in 105ms                              
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + anyio==4.15.1
 + attrs==26.1.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + datasets==5.0.1
 + dill==0.4.1
 + filelock==4.0.9
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.33.0
 + idna==3.20
 + multidict==6.9.1
 + multiprocess==0.70.19
 + numpy==2.5.3
 + packaging==26.3
 + pandas==3.0.6
 + propcache==0.5.4
 + pyarrow==25.0.1
 + python-dateutil==2.9.0.post0
 + pyyaml==6.0.3
 + regex==2026.9.29
 + requests==2.34.2
 + setuptools==84.0.0
 + six==1.17.0
 + tiktoken==0.14.0
 + tqdm==4.70.1
 + typing-extensions==4.16.0
 + urllib3==2.8.0
 + xxhash==4.0.1
 + yarl==1.25.1
venv OK: torch 2.11.0+cu130 | triton 3.6.0 | Tesla T4


In [4]:
# 4 x 100M-token train shards + the 1M-token validation shard (~800 MB, from kjj0/fineweb10B-gpt2).
!.venv/bin/python data/cached_fineweb10B.py 4
!du -sh data/fineweb10B && ls -la data/fineweb10B

954M	data/fineweb10B
total 976596
drwxr-xr-x 3 root root      4096 Oct  2 08:12 .
drwxr-xr-x 3 root root      4096 Oct  2 08:12 ..
drwxr-xr-x 3 root root      4096 Oct  2 08:12 .cache
-rw-r--r-- 1 root root 200001024 Oct  2 08:12 fineweb_train_000001.bin
-rw-r--r-- 1 root root 200001024 Oct  2 08:12 fineweb_train_000002.bin
-rw-r--r-- 1 root root 200001024 Oct  2 08:12 fineweb_train_000003.bin
-rw-r--r-- 1 root root 200001024 Oct  2 08:12 fineweb_train_000004.bin
-rw-r--r-- 1 root root 200001024 Oct  2 08:12 fineweb_val_000000.bin


In [ ]:
# Smoke run: the whole pipeline (warmup, stages, sampled softmax, validation, tail averaging,
# final masked eval) on a compressed 8+52+20-step schedule. Expect a few minutes.
import os

os.environ.update(NUM_SCHEDULED_ITERATIONS="8", PYTHONUNBUFFERED="1")
!.venv/bin/python train_gpt.py

logs/a20b2be7-050c-4a52-a60a-a4677dc7911c.txt
Warming up kernels on steps 0-1 (Triton JIT, SDPA, the fused optimizer), then resetting state
W1002 11:03:21.471000 45896 torch/_inductor/utils.py:1731] [1/0] Not enough SMs to use max_autotune_gemm mode
Resetting Model
Memory after warmup: 5.05 GiB allocated, 12.05 GiB reserved
step:0/80 val_loss:10.8305 val_time:36824ms train_time:310ms step_avg:309.89ms
step:5/80 train_time:74583ms step_avg:14916.54ms loss:9.7725 | data:0.1s cand:0.1s fwd:20.4s bwd:51.1s opt:2.6s other:0.3s
step:10/80 train_time:217446ms step_avg:21744.57ms loss:6.7785 | data:0.2s cand:0.3s fwd:58.0s bwd:153.3s opt:5.3s other:0.3s
step:15/80 train_time:357220ms step_avg:23814.66ms loss:6.2027 | data:0.2s cand:0.5s fwd:92.6s bwd:255.6s opt:8.0s other:0.3s
step:20/80 train_time:496693ms step_avg:24834.67ms loss:5.9123 | data:0.2s cand:0.7s fwd:127.2s bwd:357.5s opt:10.7s other:0.3s
step:25/80 train_time:636440ms step_avg:25457.61ms loss:5.6483 | data:0.3s cand:0.9s fwd:161

The smoke run's last lines are the final (canonical-masked) validation — that is the pipeline
working end to end; its `val_loss` is meaningless (8 steps).

The next cell starts the **real run**: 1,080 + 52 growth + 20 extension steps ≈ 287M tokens,
validation every 150 steps, final validation on the tail-averaged weights.
**Keep this tab open** — a disconnected runtime kills the run. Watch for `val_loss` trending
toward ≤3.5; `step_avg` in each log line gives the ETA.

In [ ]:
os.environ.update(NUM_SCHEDULED_ITERATIONS="1080", PYTHONUNBUFFERED="1")
!.venv/bin/python train_gpt.py

In [ ]:
# After the run: the full source-embedded log, and (optionally) keep a copy on Google Drive.
!ls -la logs/
# !cp -r logs /content/drive/MyDrive/nanogpt-colab-speedrun-logs

In [13]:
!nvidia-smi

Fri Oct  2 11:02:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             33W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----